In [ ]:
#| default_exp cdp

# CDP — network sniffing and replay

The cdp module wraps Chrome's DevTools Protocol behind a synchronous interface. `automation_browser()` is the single entry point — it launches Chrome with a persistent debug profile so cookies and sessions survive across runs. For SSO or enterprise sites, log in once in the browser window; every subsequent call picks up that session automatically.

Use cdp when scrapling's stealth mode is not enough: the site checks for enterprise SSO, reads cookies from a previous session, or ties requests to a specific browser fingerprint.

In [ ]:
#| export
import asyncio, json, base64, time, subprocess, sys, shutil, os, threading, httpx, socket, tempfile
from fastcore.all import patch, store_attr, AttrDict, Path
from fastcdp.core import CDP
from fossick.core import save_path as _cache, compile_pattern, paginate_api, to_md, get_options, fetch, fetch_all

In [ ]:
#| export
@patch
async def _record_loop(self:CDP, fps, quality, sid):
    "Receive screencast frames and pipe to ffmpeg, duplicating frames to maintain real-time playback"
    i = 1.0 / fps
    async with self.on('Page.screencastFrame') as q:
        await self.page.startScreencast(sid=sid, format='jpeg', quality=quality, everyNthFrame=1)
        lt,ld = time.monotonic(), None
        try:
            while self._ffmpeg.poll() is None:
                try:
                    frame = await asyncio.wait_for(q.get(), timeout=i)
                    p = frame['params']
                    data = base64.b64decode(p['data'])
                    now = time.monotonic()
                    if ld:
                        n_dup = max(1, round((now - lt) / i))
                        for _ in range(n_dup): self._ffmpeg.stdin.write(ld)
                    ld, lt = data, now
                    await self.page.screencastFrameAck(sid=sid, sessionId=p['sessionId'])
                except asyncio.TimeoutError:
                    if ld:
                        self._ffmpeg.stdin.write(ld)
                        lt = time.monotonic()
        except asyncio.CancelledError:
            if ld: self._ffmpeg.stdin.write(ld)
        except BrokenPipeError: pass

In [ ]:
#| export
@patch
async def start_recording(self:CDP, path=None, fps=25, quality=80, sid=None):
    "Start recording the active page to an mp4 file via ffmpeg"
    path = path or _cache('recording.mp4')
    self._ffmpeg = subprocess.Popen(
        ['ffmpeg', '-y', '-f', 'image2pipe', '-framerate', str(fps), '-i', '-',
         '-c:v', 'libsvtav1', '-crf', '35', '-preset', '8', '-movflags', '+faststart', path],
        stdin=subprocess.PIPE, stderr=subprocess.PIPE)
    self._rec_task = asyncio.create_task(self._record_loop(fps, quality, sid))

In [ ]:
#| export
@patch
async def stop_recording(self:CDP, sid=None):
    "Stop recording and finalize the mp4 file"
    try: await self.page.stopScreencast(sid=sid)
    except RuntimeError: pass
    self._rec_task.cancel()
    try: await self._rec_task
    except asyncio.CancelledError: pass
    if self._ffmpeg.stdin: self._ffmpeg.stdin.close()
    self._ffmpeg.wait()

In [ ]:
#| export
class PostCapture(AttrDict): pass

In [ ]:
#| export
async def _connect_cdp(port):
    'Try 127.0.0.1 then localhost to find the Chrome DevTools endpoint and connect'
    last_err = None
    for h in ('127.0.0.1', 'localhost'):
        try:
            async with httpx.AsyncClient() as c:
                url = (await c.get(f'http://{h}:{port}/json/version')).json()['webSocketDebuggerUrl']
            return await CDP.connect(wsconn=url)
        except Exception as e: last_err = e
    raise ConnectionRefusedError(f'no DevTools endpoint on port {port}') from last_err

In [ ]:
#| export
def _decode(b): return b.decode('utf-8', errors='replace') if isinstance(b, bytes) else b

def _parse_body(raw):
    "Parse a string body as JSON if possible, else return as-is"
    if raw is None: return None
    if isinstance(raw, (dict, list)): return raw
    try: return json.loads(raw)
    except (ValueError, TypeError): return raw

def _port_open(port):
    for host in ('localhost', '127.0.0.1'):
        try:
            with socket.create_connection((host, port), timeout=1): return True
        except OSError: pass
    return False

class CDPSession:
    "Sync-facing CDP session. Lazily connects to Chrome on first use."
    def __init__(self, port=9222, profile_dir=None):
        store_attr()
        self._cdp = None
        self._loop = asyncio.new_event_loop()
        self._thread = threading.Thread(target=self._loop.run_forever, daemon=True)
        self._thread.start()

    def _run(self, coro):
        "Submit coro to self._loop running in its background thread; block until done"
        return asyncio.run_coroutine_threadsafe(coro, self._loop).result()

    def _active_sid(self) -> str:
        "Return session ID for the active page, attaching to the first available tab if needed"
        if (res := self._run(self._cdp.active_page())) is not None: return res[1]
        pgs = [t for t in self._run(self._cdp('Target.getTargets')) if t['type'] == 'page']
        if not pgs:
            tid = self._run(self._cdp('Target.createTarget', url='about:blank'))
            if isinstance(tid, dict): tid = tid['targetId']
        else: tid = pgs[0]['targetId']
        return self._run(self._cdp.attach(tid))

    def ensure_connected(self):
        if self._cdp is None or not self._cdp.is_open:
            try: self._cdp = self._run(_connect_cdp(self.port))
            except Exception as ex:
                print(f"CDP connection failed on port {self.port}: {ex}. Attempting to launch Chrome...")
                self.launch()
                self._cdp = self._run(_connect_cdp(self.port))
        return self._active_sid()

    def close(self):
        if self._cdp and self._cdp.is_open: self._run(self._cdp.close())
        self._cdp = None

    def __enter__(self): return self
    def __exit__(self, *_): self.close()
    def __repr__(self): return f"CDPSession(port={self.port},connected={self._cdp is not None})"

In [ ]:
#| export
_plat_loc = dict(
    darwin=[Path('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'),
            Path('/Applications/Chromium.app/Contents/MacOS/Chromium')],
    win32=[Path(os.environ.get('LOCALAPPDATA','')) / 'Google/Chrome/Application/chrome.exe',
           Path(os.environ.get('PROGRAMFILES','')) / 'Google/Chrome/Application/chrome.exe',
           Path(os.environ.get('PROGRAMFILES(X86)','')) / 'Google/Chrome/Application/chrome.exe'],
    linux=['google-chrome', 'google-chrome-stable', 'chromium-browser', 'chromium'])

def _linux_loc():
    for name in _plat_loc['linux']:
        p = Path(shutil.which(name) or Path('/usr/bin')/name)
        if p.exists(): return p
    return None

def _find_chrome(chrome_path=None):
    "Find Chrome/Chromium binary across platforms; raises FileNotFoundError if absent"
    if chrome_path: return chrome_path
    sp = sys.platform
    return _linux_loc() if sp == 'linux' else next(str(p) for p in _plat_loc[sp] if p.exists())

def _default_profile_dir():
    "Platform-appropriate CDP debug profile directory"
    if sys.platform == 'darwin': return str(Path.home()/'Library/Application Support/ChromeDebug')
    if sys.platform == 'win32': return str(Path(os.environ.get('LOCALAPPDATA', str(Path.home())))/'ChromeDebug')
    return str(Path.home()/'.config/chrome-debug')

@patch
def launch(self:CDPSession, wait=10, chrome_path=None) -> 'CDPSession':
    "Launch Chrome with remote debugging on self.port"
    chrome = _find_chrome(chrome_path)
    args = [chrome, f'--remote-debugging-port={self.port}',
            f'--user-data-dir={self.profile_dir or _default_profile_dir()}']
    subprocess.Popen(args, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(wait):
        time.sleep(1)
        if _port_open(self.port): return self
    return self

In [ ]:
#| export
from importlib.resources import files as _pkg_files

In [ ]:
#| export
def _launchd_registered(label): return subprocess.run(['launchctl', 'list', label], capture_output=True).returncode == 0

def _get_chrome_info(port:int=9222, chrome_path=None, profile_dir=None):
    "Return a dict of info needed to launch Chrome with CDP on macOS"
    return AttrDict(port=port, chrome=_find_chrome(chrome_path), label=f'com.fossick.chrome-debug-{port}',
        profile_dir=profile_dir or _default_profile_dir(), origins=f'http://localhost:{port},http://127.0.0.1:{port}')

def _mac_chrome(port:int=9222, chrome_path=None, profile_dir=None, uninstall:bool=False):
    "Set up a launchd agent to start Chrome with CDP on login. Idempotent. macOS only."
    ci, domain = _get_chrome_info(port, chrome_path, profile_dir), f'gui/{os.getuid()}'
    plist = Path.home() / f'Library/LaunchAgents/{ci.label}.plist'
    if uninstall:
        subprocess.run(['launchctl', 'bootout', f'{domain}/{ci.label}'], check=False)
        plist.unlink(missing_ok=True)
        return True
    log = str(Path.home() / f'Library/Logs/chrome-debug-{port}.log')
    pp = _pkg_files('fossick').joinpath('chrome_debug.plist')
    t = pp.read_text().format(log=log, **ci)
    plist.mk_write(t)
    if _launchd_registered(ci.label):
        subprocess.run(['launchctl', 'bootout', f'{domain}/{ci.label}'], check=False)
        for _ in range(20):
            time.sleep(0.1)
            if not _launchd_registered(ci.label): break
    r = subprocess.run(['launchctl', 'bootstrap', domain, str(plist)])
    if r.returncode == 5:
        # Service still in bootstrap namespace (e.g. prior interrupted run); force bootout and retry
        subprocess.run(['launchctl', 'bootout', f'{domain}/{ci.label}'], check=False)
        time.sleep(1.0)
        subprocess.run(['launchctl', 'bootstrap', domain, str(plist)], check=True)
    elif r.returncode:
        raise subprocess.CalledProcessError(r.returncode, r.args)
    return True

def _linux_chrome(port:int=9222, chrome_path=None, profile_dir=None, uninstall:bool=False):
    "Set up a systemd user service to start Chrome with CDP on login. Idempotent. Linux only."
    ci = _get_chrome_info(port, chrome_path, profile_dir)
    svc = Path.home() / f'.config/systemd/user/chrome-debug-{port}.service'
    if uninstall:
        subprocess.run(['systemctl', '--user', 'disable', '--now', svc.stem], check=False)
        svc.unlink(missing_ok=True)
        return True
    svc.mk_write(_pkg_files('fossick').joinpath('chrome_debug.service').read_text().format(**ci))
    subprocess.run(['systemctl', '--user', 'enable', '--now', svc.stem], check=True)
    return True

def _win_chrome(port:int=9222, chrome_path=None, profile_dir=None, uninstall:bool=False):
    "Set up a scheduled task to start Chrome with CDP on login. Idempotent. Windows only."
    ci, task = _get_chrome_info(port, chrome_path, profile_dir), f'ChromeDebug-{port}'
    if uninstall:
        subprocess.run(['schtasks', '/delete', '/tn', task, '/f'], check=False)
        return True
    with tempfile.NamedTemporaryFile(suffix='.xml', delete=False, mode='w', encoding='utf-8') as f:
        f.write(_pkg_files('fossick').joinpath('chrome_debug_task.xml').read_text().format(**ci))
        tmp = f.name
    subprocess.run(['schtasks', '/create', '/tn', task, '/xml', tmp, '/f'], check=True)
    Path(tmp).unlink(missing_ok=True)
    return True

def setup_chrome_daemon(port:int=9222, chrome_path=None, profile_dir=None, uninstall:bool=False) -> bool:
    "Install (or uninstall) a system service that starts Chrome with remote debugging at login"
    sp = sys.platform
    fn = _mac_chrome if sp == 'darwin' else _linux_chrome if sp == 'linux' else _win_chrome if sp == 'win32' else None
    if fn: return fn(port, chrome_path, profile_dir, uninstall)
    else: raise NotImplementedError(f'daemon setup not supported on {sp}')

In [ ]:
s = CDPSession(port=9222)
assert s._cdp is None and s.port == 9222
s2 = CDPSession(port=9223, profile_dir='/tmp/chrome')
assert s2.port == 9223 and s2.profile_dir == '/tmp/chrome'
assert '9222' in repr(s) and 'False' in repr(s)
assert isinstance(_default_profile_dir(), str)
assert _find_chrome('/usr/bin/chrome') == '/usr/bin/chrome'

In [ ]:
#| export
async def _sniff_async(cdp, pattern, timeout, url, posts_only, record, record_path, sid, screenshot_path):
    staged, shot_n, rx = {}, [0], compile_pattern(pattern)
    await cdp.network.enable(sid=sid)
    if record: await cdp.start_recording(record_path, sid=sid)
    async def _handle_req(msg):
        p = msg['params']
        req = p.get('request', {})
        m = req.get('method', '')
        if posts_only and m != 'POST': return
        ru = req.get('url', '')
        if not rx.search(ru): return
        rid = p['requestId']
        staged[rid] = dict(url=ru, method=m, request_headers=dict(req.get('headers', {})), screenshot_path=None,
            request_body=_parse_body(req.get('postData')), response_body=None, timestamp=p.get('timestamp', 0.0))
        if screenshot_path:
            try:
                res = await cdp.page.captureScreenshot(format='png', sid=sid)
                base, n = Path(screenshot_path), shot_n[0]
                name = f'{base.stem}_{n}{base.suffix or '.png'}'
                shot_n[0] += 1
                sp = (base.parent / name) if base.is_absolute() else _cache(name)
                sp.write_bytes(base64.b64decode(res))
                staged[rid]['screenshot_path'] = str(sp)
            except Exception as e: print(f"screenshot failed: {e}")

    async with cdp.on('Network.requestWillBeSent') as req_q:
        async with cdp.on('Network.loadingFinished') as done_q:
            if url:
                nav = asyncio.create_task(cdp.goto(url, sid=sid, timeout=timeout))
                while not nav.done():
                    await asyncio.sleep(0.1)
                    while not req_q.empty(): await _handle_req(req_q.get_nowait())
                try: await nav
                except Exception: pass
            else:
                end = time.monotonic() + timeout
                while time.monotonic() < end:
                    await asyncio.sleep(0.1)
                    while not req_q.empty(): await _handle_req(req_q.get_nowait())
            await asyncio.sleep(0.5)
            while not req_q.empty(): await _handle_req(req_q.get_nowait())
            while not done_q.empty():
                msg = done_q.get_nowait()
                rid = msg['params']['requestId']
                if rid not in staged: continue
                try:
                    resp = await cdp.network.getResponseBody(requestId=rid, sid=sid)
                    body = resp.get('body', '')
                    if resp.get('base64Encoded'): body = base64.b64decode(body).decode('utf-8', errors='replace')
                    staged[rid]['response_body'] = _parse_body(body)
                except Exception: pass
    if record: await cdp.stop_recording(sid=sid)
    return [PostCapture(**d) for d in staged.values()]

In [ ]:
#| export
@patch
def sniff(self:CDPSession, pattern='.*', timeout=30, url=None,
          record=False, record_path=None, posts_only=False,
          screenshot_path=None) -> list:
    "Sniff HTTP requests from Chrome. Passive (user browses) if url=None, active if url given."
    sid = self.ensure_connected()
    if record and record_path is None: record_path = str(_cache('recording.mp4'))
    return self._run(_sniff_async(self._cdp, pattern, timeout, url, posts_only, record, record_path, sid, screenshot_path))

In [ ]:
#| export
@patch
def to_paginate_args(self:CDPSession, capture:PostCapture) -> dict:
    "Convert a PostCapture into kwargs ready to unpack into paginate_api()"
    return dict(url=capture.url, payload=capture.request_body, method='POST', headers=capture.request_headers)

@patch
def replay(self:CDPSession, cap:PostCapture, body:dict=None, headers:dict=None) -> dict:
    "Replay a captured request with optional body/header overrides. Returns {url, status, html, data}."
    merged_body = {**cap.request_body, **(body or {})} if isinstance(cap.request_body, dict) else cap.request_body
    return fetch(cap.url, method='POST', payload=merged_body, headers={**cap.request_headers, **(headers or {})})

In [ ]:
#| export
@patch
def goto(self:CDPSession, url:str, timeout:int=30) -> None:
    "Navigate the active page to url"
    sid = self.ensure_connected()
    async def _nav():
        await self._cdp.page.navigate(url=url, sid=sid)
        end = time.monotonic() + timeout
        while time.monotonic() < end:
            await asyncio.sleep(0.5)
            try:
                r = await self._cdp.runtime.evaluate(expression='document.readyState', returnByValue=True, sid=sid)
                if (r.get('value', '') if isinstance(r, dict) else r) == 'complete': break
            except: break
    self._run(_nav())

In [ ]:
#| export
@patch
def source(self:CDPSession) -> dict:
    "Return current page as {'html': ..., 'url': ...} — pass directly to to_md()"
    sid = self.ensure_connected()
    async def _get():
        def _val(r): return r.get('value', '') if isinstance(r, dict) else (r or '')
        html = await self._cdp.runtime.evaluate(expression='document.documentElement.outerHTML', returnByValue=True, sid=sid)
        url = await self._cdp.runtime.evaluate(expression='window.location.href', returnByValue=True, sid=sid)
        return {'html': _val(html), 'url': _val(url)}
    return self._run(_get())

In [ ]:
cap = PostCapture(url='https://api.danmurphys.com.au/apis/ui/Search/products',
    method='POST', request_headers={'content-type': 'application/json', 'x-api-key': 'abc'},
    request_body={'pageNumber': 1, 'pageSize': 48, 'query': 'wine'},
    response_body={'Items': [{'Name': 'Test Wine'}]}, timestamp=time.time())
s = CDPSession(port=9222)
args = s.to_paginate_args(cap)
assert args['url'] == cap.url and args['method'] == 'POST'
assert args['payload'] == cap.request_body and args['headers'] == cap.request_headers
cap2 = PostCapture(url='https://t.com', method='POST', request_headers={},
                   request_body=[{'id': 1}], response_body=None, timestamp=time.time())
assert cap2.request_body == [{'id': 1}]

In [ ]:
#| export
def automation_browser(port=9222, profile_dir=None) -> 'CDPSession':
    "CDPSession with a persistent debug profile — sessions and cookies survive across runs"
    return CDPSession(port=port, profile_dir=profile_dir)

In [ ]:
#| export
@patch
def screenshot(self:CDPSession, url:str=None, path=None, full_page:bool=False) -> str:
    """Navigate to url (if given) then capture a PNG screenshot. Returns the saved file path.
    If full_page=True, resizes the viewport to the full page height before capture."""
    sid = self.ensure_connected()
    if url: self.goto(url, timeout=30)
    async def _capture():
        if full_page:
            layout = await self._cdp.page.getLayoutMetrics(sid=sid)
            cs = layout.get('contentSize', layout.get('cssContentSize', {}))
            w, h = int(cs.get('width', 1280)), int(cs.get('height', 800))
            await self._cdp.emulation.setDeviceMetricsOverride(
                width=w, height=h, deviceScaleFactor=1, mobile=False, sid=sid)
        data = await self._cdp.page.captureScreenshot(format='png', sid=sid)
        return base64.b64decode(data)
    img_bytes = self._run(_capture())
    out = Path(path) if path else _cache(f'screenshot_{int(time.time()*1000)}.png')
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    Path(out).write_bytes(img_bytes)
    return str(out)

In [ ]:
assert setup_chrome_daemon(port=9222) is True
assert setup_chrome_daemon(port=9222, uninstall=True) is True

## Integration Examples

The first example navigates to a site with a headless browser, captures the POST requests the page makes, then replays one to paginate through all results without keeping the browser open. The second shows the same workflow using scrapling's `fetch_all()` for pages that do not need JavaScript.

In [ ]:
#| eval: false
# automation_browser starts Chrome with a persistent profile on port 9222
with automation_browser() as s:
    caps = s.sniff(url='https://www.danmurphys.com.au/wine', pattern='*api.danmurphys.com.au/apis*', timeout=10,
                   posts_only=True, screenshot_path='dm_sniff.png')
    print(f"Captured {len(caps)} POST(s):")
    if caps:
        all_items = paginate_api(
            **s.to_paginate_args(caps[0]),
            results_field='Items',
            size_field='pageSize',
            page_field='pageNumber',
            max_pages=2
        )
        print(f"\nTotal products: {len(all_items)}")
        print("Sample:", all_items[0]['Name'].strip() if all_items else 'none')

Captured 0 POST(s):


In [ ]:
#| eval: false
# Approach A: discover sargas from the page dropdown, fetch all in parallel
BASE = 'https://www.valmiki.iitk.ac.in/sloka'
first = fetch(f'{BASE}?field_kanda_tid=1&language=dv&field_sarga_value=1')
sopts = get_options(first, '#edit-field-sarga-value')
print(f"Found {len(sopts)} sargas: {[o['value'] for o in sopts[:5]]}...")
urls = [f'{BASE}?field_kanda_tid=1&language=dv&field_sarga_value={o["value"]}' for o in sopts[:5]]
pages = fetch_all(urls, sel='.view-content')
shlokas = {p['url']: to_md(p) for p in pages if p['status'] == 200}
print(f"Fetched {len(shlokas)} sargas")
if shlokas: print(next(iter(shlokas.values()))[:200])

[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=1> (referer: https://www.google.com/)


Found 77 sargas: ['1', '2', '3', '4', '5']...


[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=5> (referer: https://www.google.com/)
[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=3> (referer: https://www.google.com/)
[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=2> (referer: https://www.google.com/)
[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=4> (referer: https://www.google.com/)
[2026-06-03 08:12:49] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=1> (referer: https://www.google.com/)


Fetched 5 sargas
[Saint Narada visits hermitage of Valmiki -- Valmiki queries about a single perfect individual bestowed with all good qualities enumerated by him -- Narada, knower of past, present and future, identif


### Enterprise / SSO sites

`automation_browser` uses a persistent profile — log in once in the browser window and subsequent runs reuse those cookies automatically. Works with any site that ties auth to the browser session: Jira, Confluence, LinkedIn, internal tools.

In [ ]:
#| eval: false
# Log in once in the browser; subsequent runs reuse the session automatically.
# Read any SSO-protected page as markdown
with automation_browser() as s:
    s.goto('https://www.linkedin.com/feed/')
    print(to_md(s.source())[:2000])

## 0 notifications

Skip to main contentSkip to sidebarSkip to primary contentSkip to aside

  * Home
  * My Network
  * Jobs
  * Messaging
  * 4Notifications
  * Me

* * *

  * For Business

Reactivate Premium: 50% Off

Karthik Rajgopal Laxmi Naarayanan

Director of Machine learning at Bain

Greater Melbourne Area

Bain & Company

Boost your job search

Reactivate Premium: 50% Off

Profile viewers

138

Post impressions

39

Saved items

Groups

Newsletters

Events

Start a post

Video

Photo

Write article

* * *

Sort by: **Top**

New posts

## Feed post

Suggested

* * *

Jacob👋 Voytko

• 3rd+

Staff Backend Engineer

18h • Edited •

Follow

Aaron Levie, the CEO of Box, recently tweeted that CEOs are uniquely prone to AI psychosis, the type where they are delusional about what can be accomplished with an LLM, because CEOs do not perform the "last mile" of work. They never see the effort required to coax LLMs to perform useful work. They just see the happy path.  
  
So I did my par

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()